# 04 — Reward Engineering : concevoir de bonnes récompenses

Le plus grand défi du RL n'est pas l'algorithme — c'est de définir **ce qu'on veut vraiment récompenser**.

Ce notebook explore le **reward shaping** : comment guider l'agent avec des récompenses bien conçues.

Source : https://pythonprogramming.net/engineering-rewards-reinforcement-learning-stable-baselines-3-tutorial/

## Le problème fondamental : l'exploration

Un agent apprend à partir de l'expérience. Mais si la récompense est **rare**, il ne la découvre peut-être jamais par hasard.

**Exemple Snake :**
- La récompense arrive seulement quand l'agent mange une pomme
- Un agent aléatoire mange rarement une pomme
- Sans jamais voir de récompense positive, l'agent ne sait pas dans quelle direction s'améliorer

**Solution** : donner des récompenses intermédiaires pour guider l'agent vers le bon comportement.

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_checker import check_env
import os

os.makedirs("logs", exist_ok=True)
print("Imports OK")

## Environnement de base : navigation vers une cible

On reprend l'environnement du notebook 03, mais on va expérimenter différentes fonctions de récompense.

In [ ]:
class BaseNavEnv(gym.Env):
    """Environnement de navigation. La fonction de reward est à implémenter dans les sous-classes."""

    SIZE = 10
    MAX_STEPS = 200

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(
            low=-self.SIZE, high=self.SIZE, shape=(2,), dtype=np.float32
        )
        self._step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.agent_pos = self.np_random.integers(0, self.SIZE, size=2).astype(np.float32)
        self.target_pos = self.np_random.integers(0, self.SIZE, size=2).astype(np.float32)
        while np.array_equal(self.agent_pos, self.target_pos):
            self.target_pos = self.np_random.integers(0, self.SIZE, size=2).astype(np.float32)
        self._step_count = 0
        return self._get_obs(), {}

    def step(self, action):
        moves = {0: [0,1], 1: [0,-1], 2: [-1,0], 3: [1,0]}
        self.agent_pos = np.clip(
            self.agent_pos + moves[action], 0, self.SIZE - 1
        ).astype(np.float32)
        self._step_count += 1

        reached = np.array_equal(self.agent_pos, self.target_pos)
        terminated = reached
        truncated = self._step_count >= self.MAX_STEPS

        reward = self._compute_reward(reached)
        return self._get_obs(), reward, terminated, truncated, {}

    def _get_obs(self):
        return (self.target_pos - self.agent_pos).astype(np.float32)

    def _compute_reward(self, reached):
        raise NotImplementedError

print("Environnement de base défini")

## Approche 1 : Récompense binaire (naïve)

Récompense seulement quand la cible est atteinte. L'agent ne reçoit aucun signal intermédiaire.

In [ ]:
class BinaryRewardEnv(BaseNavEnv):
    """Récompense binaire : +10 si cible atteinte, 0 sinon."""

    def _compute_reward(self, reached):
        return 10.0 if reached else 0.0


check_env(BinaryRewardEnv())

env = BinaryRewardEnv()
model_binary = PPO('MlpPolicy', env, verbose=0, tensorboard_log="logs")
model_binary.learn(total_timesteps=50_000, tb_log_name="1_binary")

mean, std = evaluate_policy(model_binary, BinaryRewardEnv(), n_eval_episodes=50)
print(f"Récompense binaire → reward moyen : {mean:.3f} ± {std:.3f}")
env.close()

## Approche 2 : Pénalité de survie

Ajouter `-0.1` à chaque step pour pousser l'agent à atteindre la cible rapidement.

Intuition : si chaque step coûte, l'agent préfère les chemins courts.

In [ ]:
class SurvivalPenaltyEnv(BaseNavEnv):
    """Récompense binaire + pénalité par step pour encourager l'efficacité."""

    def _compute_reward(self, reached):
        if reached:
            return 10.0
        return -0.1  # chaque step sans succès coûte


check_env(SurvivalPenaltyEnv())

env = SurvivalPenaltyEnv()
model_penalty = PPO('MlpPolicy', env, verbose=0, tensorboard_log="logs")
model_penalty.learn(total_timesteps=50_000, tb_log_name="2_survival_penalty")

mean, std = evaluate_policy(model_penalty, SurvivalPenaltyEnv(), n_eval_episodes=50)
print(f"Pénalité de survie → reward moyen : {mean:.3f} ± {std:.3f}")
env.close()

## Approche 3 (échouée) : Pénalité de distance euclidienne

Idée intuitive : pénaliser proportionnellement à la distance à la cible.

**Problème** : la distance est toujours négative. L'agent apprend que *vivre* est douloureux → il préfère terminer l'épisode immédiatement (même sans atteindre la cible).

In [ ]:
class NaiveDistancePenaltyEnv(BaseNavEnv):
    """Pénalité = distance euclidienne à la cible. Résultat : l'agent veut mourir."""

    def _compute_reward(self, reached):
        if reached:
            return 10.0
        dist = np.linalg.norm(self.target_pos - self.agent_pos)
        # La distance est TOUJOURS positive → récompense TOUJOURS négative
        # L'agent minimise la douleur en terminant vite (truncated)
        return -dist


env = NaiveDistancePenaltyEnv()
model_naive = PPO('MlpPolicy', env, verbose=0, tensorboard_log="logs")
model_naive.learn(total_timesteps=50_000, tb_log_name="3_naive_distance")

mean, std = evaluate_policy(model_naive, NaiveDistancePenaltyEnv(), n_eval_episodes=50)
print(f"Pénalité distance naïve → reward moyen : {mean:.3f} ± {std:.3f}")
print("(probablement très mauvais : l'agent préfère s'arrêter plutôt que de chercher la cible)")
env.close()

## Approche 4 (réussie) : Distance avec offset positif

**Solution du tutoriel** : utiliser un offset pour que la récompense soit positive, plus grande quand on est proche.

```
reward = (offset - distance) / normalisation
```

- Quand distance = 0 (cible atteinte) → reward = offset/normalisation (maximum)
- Quand distance = offset → reward = 0
- On ajoute un **gros bonus** pour la récompense finale

L'agent a maintenant un signal positif qui augmente à mesure qu'il s'approche.

In [ ]:
class ShapedRewardEnv(BaseNavEnv):
    """
    Reward shaping : proximité récompensée progressivement.
    Inspiré du tutoriel Snake (adaptation à la grille 10x10).
    """

    # La distance max sur une grille 10x10 est ~14 (diagonale)
    # L'offset doit être > distance_max pour que la reward reste positive
    OFFSET = 20.0
    NORMALISATION = 5.0
    BONUS_CIBLE = 100.0  # gros bonus quand la cible est atteinte

    def _compute_reward(self, reached):
        dist = np.linalg.norm(self.target_pos - self.agent_pos)
        proximity_reward = (self.OFFSET - dist) / self.NORMALISATION

        if reached:
            return (proximity_reward + self.BONUS_CIBLE) / self.NORMALISATION
        return proximity_reward / self.NORMALISATION


check_env(ShapedRewardEnv())

env = ShapedRewardEnv()
model_shaped = PPO('MlpPolicy', env, verbose=0, tensorboard_log="logs")
model_shaped.learn(total_timesteps=50_000, tb_log_name="4_shaped")

mean, std = evaluate_policy(model_shaped, ShapedRewardEnv(), n_eval_episodes=50)
print(f"Reward shaping → reward moyen : {mean:.3f} ± {std:.3f}")
env.close()

## Comparaison des approches

In [ ]:
print("=" * 60)
print(f"{'Approche':<30} {'Reward moyen':>15} {'Std':>10}")
print("=" * 60)

results = [
    ("1. Binaire (+10 si cible)",    model_binary,  BinaryRewardEnv),
    ("2. Pénalité de survie (-0.1)", model_penalty, SurvivalPenaltyEnv),
    ("3. Distance naïve (échoue)",   model_naive,   NaiveDistancePenaltyEnv),
    ("4. Distance avec offset",      model_shaped,  ShapedRewardEnv),
]

for name, model, env_cls in results:
    e = env_cls()
    mean, std = evaluate_policy(model, e, n_eval_episodes=50)
    e.close()
    print(f"{name:<30} {mean:>15.3f} {std:>10.3f}")

## Visualiser la structure des récompenses

Comprendre ce que "voit" l'agent depuis différentes positions.

In [ ]:
# Simuler le signal de récompense de ShapedRewardEnv selon la distance
distances = np.linspace(0, 15, 100)
OFFSET = 20.0
NORMALISATION = 5.0
BONUS = 100.0

rewards_shaped = (OFFSET - distances) / NORMALISATION / NORMALISATION
rewards_naive = -distances

print("Distance | Reward naïf | Reward shapé")
print("-" * 40)
for d in [0, 2, 5, 10, 14]:
    naive = -d
    shaped = (OFFSET - d) / NORMALISATION / NORMALISATION
    print(f"{d:>8.0f} | {naive:>11.2f} | {shaped:>12.3f}")

print()
print("Observation clé :")
print("  Naïf  : reward TOUJOURS négatif → l'agent fuit")
print("  Shapé : reward positif proche de la cible → l'agent est guidé")

## Principes du Reward Engineering

### 1. Éviter les récompenses toujours négatives
Si l'agent reçoit toujours des pénalités, il cherche à terminer l'épisode (mourir = fin de la douleur).

### 2. Signal intermédiaire pour guider l'exploration
Sans signal, l'agent doit trouver la récompense par hasard. Plus la récompense est rare, plus c'est difficile.

### 3. Conserver l'incitatif principal
Le bonus intermédiaire ne doit pas éclipser l'objectif final :
- Si se rapprocher rapporte 0.4/step et atteindre la cible rapporte 0.5 → l'agent peut ignorer la cible !
- Le bonus final doit rester significativement plus grand

### 4. Normaliser les récompenses
Des récompenses à des échelles très différentes (0.01 vs 10000) déstabilisent l'entraînement.

### 5. Attention au reward hacking
L'agent optimise exactement ce qu'on mesure, pas ce qu'on veut vraiment :
- Récompenser le score → l'agent exploite des bugs
- Récompenser la vitesse → l'agent prend des raccourcis dangereux

| Anti-pattern | Symptôme | Correction |
|---|---|---|
| Reward toujours négatif | Agent se suicide | Offset positif |
| Pas de signal intermédiaire | Agent n'apprend pas | Reward de proximité |
| Bonus final trop faible | Agent ignore l'objectif | Augmenter le bonus final |
| Échelles très différentes | Instabilité d'entraînement | Normalisation |

## Résumé

La conception des récompenses est souvent **plus importante** que le choix de l'algorithme.

Checklist avant d'entraîner :
- [ ] La récompense est-elle toujours négative ? → Ajouter un offset
- [ ] L'agent peut-il découvrir la récompense principale par hasard ? → Ajouter un signal intermédiaire
- [ ] Le bonus intermédiaire est-il proportionnel au bonus final ? → Vérifier l'échelle
- [ ] L'agent peut-il tricher pour maximiser la récompense ? → Tester des comportements adversariaux

**Tu as maintenant toutes les bases pour :**
1. Comprendre le RL et ses composants (`01_introduction_rl.ipynb`)
2. Sauvegarder et suivre l'entraînement (`02_save_load_tensorboard.ipynb`)
3. Créer ton propre environnement (`03_custom_environment.ipynb`)
4. Concevoir de bonnes récompenses (`04_reward_engineering.ipynb`) ← tu es ici